# Optimizing LibreYOLO for Qualcomm® Snapdragon Devices Using SNPE

This notebook walks through the full pipeline to optimize a **LibreYOLO** object detection model for efficient inference on Qualcomm® Snapdragon devices using the **Snapdragon Neural Processing Engine (SNPE) SDK**.

Note: This notebook uses the SNPE command-line flow because it is still widely used for DLC-based deployment. For newer projects, Qualcomm’s QAIRT tools provide a more unified interface around conversion, quantization, execution, and profiling.

The pipeline covers:
1. **Image preprocessing** — resize and normalize images for model input.
2. **Calibration dataset preparation** — generate representative `.raw` samples for INT8 quantization.
3. **Model export** — load a pre-trained LibreYOLOXs checkpoint and export it to ONNX.
4. **DLC conversion** — convert the ONNX model to SNPE’s Deep Learning Container (DLC) format.
5. **INT8 quantization** — apply post-training quantization for faster, more efficient inference.
6. **HTP graph compilation** — compile the DLC offline for the Hexagon Tensor Processor (HTP) on the Snapdragon 778G (sm7325).

We begin by importing the necessary libraries.

In [1]:
# Import necessary libraries.
import cv2
import glob
import os
import onnx
import random
import torch
import uuid

import numpy as np
import onnx.shape_inference

from libreyolo.models.yolox.nn import LibreYOLOXModel
from onnx import helper, TensorProto

## Preprocessing and Calibration Data

Before converting or quantizing the model, two things are needed:

- A **preprocessing function** to transform raw images into the tensor format expected by LibreYOLO (letterbox-resized to 640×640, BGR color, 0–255 float32, CHW layout).
- A **representative calibration dataset** in SNPE's `.raw` binary format. SNPE uses this dataset during post-training quantization to compute per-layer scale factors for INT8 weights and activations.

The cell below defines the `letterbox()` and `preprocess()` helper functions used throughout this pipeline.

In [2]:
def letterbox(image: np.ndarray, target: int = 640, pad_value: int = 114) -> tuple:
    """Resize image to (target×target) with top-left letterbox padding.

    YOLOX preprocessing: image is placed at top-left (0, 0); padding fills
    the right and bottom edges. Color format stays BGR, values stay 0–255.

    Args:
        image (np.ndarray): Input BGR image as NumPy array.
        target (int): Target image size (square).
        pad_value (int): Padding pixel value (default 114, YOLOX convention).

    Returns:
        padded (np.ndarray): Padded image, float32 HWC BGR, 0-255 range.
        ratio (float): Scale factor applied to original dimensions.
    """
    h, w = image.shape[:2]
    ratio = min(target / h, target / w)
    new_w, new_h = int(w * ratio), int(h * ratio)

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    padded = np.full((target, target, 3), pad_value, dtype=np.float32)
    padded[:new_h, :new_w] = resized

    return padded, ratio


def preprocess(original_image: np.ndarray, size: int = 640) -> np.ndarray:
    """
    Preprocess the input image for model inference.

    YOLOX expects: BGR, 0-255 float32, CHW, top-left letterbox.
    No /255 normalisation and no BGR→RGB conversion.

    Args:
        original_image (np.ndarray): Input BGR image as NumPy array.
        size (int): Target image size.

    Returns:
        np.ndarray: Preprocessed float32 image, CHW layout, 0-255 range.
    """
    padded, _ = letterbox(original_image, size)
    tensor = padded.transpose(2, 0, 1)  # HWC → CHW
    return np.ascontiguousarray(tensor)

### Preparing the Calibration Dataset

SNPE's quantization tool (`snpe-dlc-quantize`) requires input samples in `.raw` format — flat binary files containing `float32` pixel values with shape `(C, H, W)`.

The code below:
1. Downloads the **COCO 2017 validation set** (~777 MB).
2. Randomly samples **2000 images** (with a fixed seed for reproducibility).
3. Preprocesses each image using `preprocess()` and saves it as a `.raw` file.
4. Generates a `filenames.txt` manifest listing all `.raw` files — required by `snpe-dlc-quantize`.

In [3]:
if not os.path.exists("val"):
    !bash -c 'curl -L -o val2017.zip http://images.cocodataset.org/zips/val2017.zip'
    !bash -c 'unzip val2017.zip -d val'
    !bash -c 'rm val2017.zip'

if not os.path.exists("raw"):
    !bash -c 'mkdir raw'

    random.seed(42)
    SAMPLE_SIZE = 2000

    filenames = glob.glob(f"val/**/*.jpg", recursive=True)
    random.shuffle(filenames)

    for filename in filenames[:SAMPLE_SIZE]:
        original_image = cv2.imread(filename)
        processed_image = preprocess(original_image)
        processed_image.tofile(f"raw/{uuid.uuid4()}.raw")

if not os.path.exists("raw/filenames.txt"):
    !bash -c 'find raw -name "*.raw" > raw/filenames.txt'

### Loading LibreYOLO and Exporting to ONNX

SNPE does not consume LibreYOLO models in PyTorch format directly. The model must first be exported to the **ONNX (Open Neural Network Exchange)** format, which SNPE can then parse and convert to its DLC format.

The code below:
1. Downloads the pre-trained **LibreYOLOXs** checkpoint from Hugging Face.
2. Loads it using the `LibreYOLO` wrapper and sets it to evaluation mode.
3. Exports it to ONNX using `torch.onnx.export()` with `opset_version=13`.
4. Applies **ONNX graph surgery** to eliminate all mixed-range intermediate tensors and produce two independent output branches.

The model decodes the grid offsets inside the graph and returns two outputs:

| Output   | Shape          | Range  | Description                              |
|----------|----------------|--------|------------------------------------------|
| `bboxes` | `[1, 8400, 4]` | 0–640  | Decoded bbox coords `[cx, cy, w, h]`     |
| `scores` | `[1, 8400, 81]`| 0–1    | Objectness + 80 class probabilities      |

where `8400 = 80×80 + 40×40 + 20×20` anchor-free grid cells.

The bbox format is `[cx, cy, w, h]` in the resized 640×640 input coordinate space. When using the model, confidence filtering, `cxcywh → xyxy` conversion, scaling back to the original image, and NMS are still required.

In [4]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../models/snpe", exist_ok=True)

if not os.path.exists("../models/LibreYOLOXs.pt"):
    !bash -c 'curl -L -o ../models/LibreYOLOXs.pt https://huggingface.co/LibreYOLO/LibreYOLOXs/resolve/main/LibreYOLOXs.pt?download=true'

checkpoint = torch.load(
    "../models/LibreYOLOXs.pt",
    map_location="cpu"
)

model = LibreYOLOXModel(config="s", nb_classes=80)
model.load_state_dict(checkpoint["model"], strict=True)

model.eval()
model.head.export = True


def _split_onnx_for_dsp(path):
    """Restructure the ONNX graph to eliminate mixed-range intermediate tensors.

    The original YOLOX export produces per-scale Concat nodes that mix decoded
    bbox coordinates (range 0-640) with sigmoid scores (range 0-1) in a single
    [1, n, 85] tensor.  With INT8 per-tensor quantization, the scale is set to
    cover 0-640, which maps all 0-1 score values to INT8 zero on DSP.

    This function removes those mixed-range Concat nodes and wires the already-
    existing [1, n, 81] score tensors directly into a separate cross-scale
    Concat, producing two independent output branches with their own INT8 scales:

        bboxes  [1, 8400, 4]  -- range 0-640   -> INT8 scale ~2.5 / step
        scores  [1, 8400, 81] -- range 0-1      -> INT8 scale ~0.004 / step

    Every new node is given an explicit name so that SNPE/QAIRT DLC layer
    names are predictable (e.g. setOutputLayers({"bboxes", "scores"})).

    The function is idempotent: if no mixed-range nodes are found, it prints a
    message and returns without modifying the file.
    """
    model = onnx.load(path)
    model = onnx.shape_inference.infer_shapes(model)
    graph = model.graph

    shape_map = {
        vi.name: [d.dim_value for d in vi.type.tensor_type.shape.dim]
        for vi in list(graph.value_info) + list(graph.input) + list(graph.output)
        if vi.type.HasField("tensor_type") and vi.type.tensor_type.shape
    }

    SCALE_NS = {6400, 1600, 400}
    per_scale, final_cat = [], None

    for node in graph.node:
        if node.op_type != "Concat":
            continue
        s = shape_map.get(node.output[0], [])
        ax = next((a.i for a in node.attribute if a.name == "axis"), None)
        if len(s) == 3 and s[0] == 1 and s[2] == 85 and s[1] in SCALE_NS and ax == -1:
            per_scale.append(node)
        elif s == [1, 8400, 85] and ax == 1:
            final_cat = node

    if not per_scale:
        print("ONNX graph already restructured, skipping.")
        return

    assert len(per_scale) == 3 and final_cat is not None, \
        "Unexpected graph structure — cannot apply surgery."

    old_to_new, new_nodes = {}, []

    for node in per_scale:
        n_k = shape_map[node.output[0]][1]
        inps = list(node.input)
        bbox_inps  = [i for i in inps if shape_map.get(i, [])[-1:] == [2]]
        score_inps = [i for i in inps if shape_map.get(i, [])[-1:] == [81]]
        bbox_name = f"_split_bboxes_{n_k}"
        new_nodes.append(helper.make_node(
            "Concat", inputs=bbox_inps, outputs=[bbox_name],
            name=bbox_name, axis=-1
        ))
        old_to_new[node.output[0]] = (bbox_name, score_inps[0])

    all_bboxes = [old_to_new[i][0] for i in final_cat.input]
    all_scores = [old_to_new[i][1] for i in final_cat.input]
    new_nodes.append(helper.make_node(
        "Concat", inputs=all_bboxes, outputs=["bboxes"],
        name="bboxes", axis=1
    ))
    new_nodes.append(helper.make_node(
        "Concat", inputs=all_scores, outputs=["scores"],
        name="scores", axis=1
    ))

    final_tensor = final_cat.output[0]
    remove_ids = {id(n) for n in per_scale} | {id(final_cat)}
    for node in graph.node:
        if final_tensor in list(node.input):
            remove_ids.add(id(node))

    new_graph_nodes = [n for n in graph.node if id(n) not in remove_ids] + new_nodes
    del graph.node[:]
    graph.node.extend(new_graph_nodes)

    while graph.output:
        graph.output.pop()
    graph.output.extend([
        helper.make_tensor_value_info("bboxes", TensorProto.FLOAT, [1, 8400, 4]),
        helper.make_tensor_value_info("scores", TensorProto.FLOAT, [1, 8400, 81]),
    ])

    model = onnx.shape_inference.infer_shapes(model)
    onnx.checker.check_model(model)
    onnx.save(model, path)
    print("ONNX graph restructured: bboxes and scores are now separate branches.")

dummy = torch.randn(1, 3, 640, 640)
onnx_path = "../models/LibreYOLOXs.onnx"

if not os.path.exists(onnx_path):
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        opset_version=13,
        dynamo=False,
        do_constant_folding=True,
        input_names=["images"],
        output_names=["detections"]
    )

    print(f"Exported decoded ONNX to: {onnx_path}.")

_split_onnx_for_dsp(onnx_path)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1343  100  1343    0     0   6127      0 --:--:-- --:--:-- --:--:--  6132
100 68.7M  100 68.7M    0     0  27.3M      0  0:00:02  0:00:02 --:--:-- 30.9M


/tmp/ipykernel_13662/184398754.py:121: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported decoded ONNX to: ../models/LibreYOLOXs.onnx.
ONNX graph restructured: bboxes and scores are now separate branches.


### Validating the ONNX output shapes

This cell checks that the exported ONNX has exactly two outputs — `bboxes` with shape `[1, 8400, 4]` and `scores` with shape `[1, 8400, 81]`. If you still see a single `detections` output with 85 columns, delete the ONNX file and re-run the previous cell so that `_split_onnx_for_dsp()` can restructure the graph.

In [5]:
onnx_model = onnx.load("../models/LibreYOLOXs.onnx")
onnx.checker.check_model(onnx_model)

print("Inputs:")
for tensor in onnx_model.graph.input:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    print(f" {tensor.name}: {dims}")

print("Outputs:")
output_shapes = {}
for tensor in onnx_model.graph.output:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    output_shapes[tensor.name] = dims
    print(f" {tensor.name}: {dims}")

assert len(onnx_model.graph.output) == 2, "Expected exactly two outputs (bboxes, scores)."
assert "bboxes" in output_shapes, "Expected output named `bboxes`."
assert "scores" in output_shapes, "Expected output named `scores`."
assert output_shapes["bboxes"] == [1, 8400, 4], f"Unexpected bboxes shape: {output_shapes['bboxes']}"
assert output_shapes["scores"] == [1, 8400, 81], f"Unexpected scores shape: {output_shapes['scores']}"

print("ONNX export is correct: bboxes [1, 8400, 4] and scores [1, 8400, 81].")

Inputs:
 images: [1, 3, 640, 640]
Outputs:
 bboxes: [1, 8400, 4]
 scores: [1, 8400, 81]
ONNX export is correct: bboxes [1, 8400, 4] and scores [1, 8400, 81].


## Converting the Model to SNPE DLC Format

SNPE uses its own model format called the **DLC (Deep Learning Container)**. The first step is to convert the ONNX model to a floating-point (**FP32**) DLC using the `snpe-onnx-to-dlc` tool. This DLC file is the starting point for all subsequent quantization and compilation steps.

In [6]:
!bash -c 'snpe-onnx-to-dlc  \
    --input_network ../models/LibreYOLOXs.onnx \
    --output_path ../models/snpe/LibreYOLOXs_fp32.dlc'

2026-05-11 08:49:45,603 - 278 - INFO - Input shape info 
2026-05-11 08:49:47,344 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-11 08:49:47,344 - 283 - WARNING - Simplified model validation failed
2026-05-11 08:49:47,715 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-11 08:49:47,715 - 283 - WARNING - Simplified model validation failed
2026-05-11 08:49:47,885 - 278 - INFO - INFO_STATIC_RESHAPE: Applying static reshape to /head/Concat_3_output_0: new name /head/Reshape_1_output_0 new shape [1, -1, 2]
2026-05-11 08:49:47,885 - 283 - WARNING - Only numerical type cast is supported. The cast op: /head/Cast will be interpreted at conversion time
2026-05-11 08:49:47,893 - 278 - INFO - INFO_STATIC_RESHAPE: Applying static reshape to /head/Concat_5_output_0: new name /head/Reshape_3_output_0 new shape [1, -1, 2]
2026-05-11 08:49:47,893 - 283 - WARNING - Only numerical type cast is supported. The cast op:

### Inspecting the FP32 DLC

The `snpe-dlc-info` command prints a detailed summary of the DLC graph, including layer names, types, input/output tensor shapes, and supported execution backends (CPU, GPU, HTP). Reviewing this output verifies that the ONNX export was clean and that all operators are supported by SNPE before proceeding to quantization.

In [7]:
!bash -c 'snpe-dlc-info  \
    --input_dlc ../models/snpe/LibreYOLOXs_fp32.dlc'

DLC info of: /workspace/models/snpe/LibreYOLOXs_fp32.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: snpe-onnx-to-dlc; unroll_gru_time_steps=True; quantization_overrides=; prepare_inputs_as_params=False; perform_axes_to_spatial_first_order=True; unroll_lstm_time_steps=True; packed_max_seq=1; validation_target=[]; packed_masked_softmax_inputs=[]; optimization_pass_mode=ir_optimizer_mainline; op_package_lib=; no_simplification=False; multi_time_steps_gru=False; match_caffe_ssd_to_tf=True; preserve_onnx_output_order=False; perform_layout_transformation=False; keep_quant_nodes=False; input_dtype=[]; keep_disconnected_nodes=False; ir_optimizer_config=; input_encoding=[]; handle_gather_negative_indices=True; copyright_file=None; force_prune_cast_ops=False; use_convert_quantization_nodes=False; package_name=None; defer_loading=False; dry_run=None; input_type=[]; float_bitwidth=32; out_

### INT8 Post-Training Quantization

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. The `snpe-dlc-quantize` tool applies **post-training quantization (PTQ)** by computing per-layer scale factors from the calibration `.raw` samples prepared earlier, producing a quantized DLC.

#### Why the ONNX graph is restructured before quantization

The original YOLOX export produces per-scale `[1, n, 85]` tensors that mix decoded bounding-box coordinates (range **0 – 640**) with sigmoid-activated scores (range **0 – 1**) in the same tensor. With default **INT8 per-tensor quantization** (256 levels), the quantizer sets a single scale to cover the full 0–640 range:

```
scale ≈ 640 / 255 ≈ 2.5  (units per step)
```

Any activation below ~1.25 rounds to INT8 `0`. Since all objectness and class-probability values lie in [0, 1], they **all collapse to zero** on DSP while CPU/GPU (float32/float16) remain unaffected.

The `_split_onnx_for_dsp()` function rewires the graph to eliminate all mixed-range intermediate tensors. The `[1, n, 81]` score tensors already exist in the graph as separate nodes (Slice outputs from the raw transposed features); the surgery routes them directly into their own cross-scale Concat without ever merging with bbox coordinates:

| Output   | Range | INT8 scale         | Values in [0, 1] map to |
|----------|-------|--------------------|-------------------------|
| `bboxes` | 0–640 | ≈ 2.5 / step       | INT8 levels 0–0 (lost)  |
| `scores` | 0–1   | ≈ 0.004 / step     | INT8 levels 0–255 ✓     |

This gives `scores` full INT8 resolution without widening any activations to INT16, preserving full DSP throughput.

In [8]:
!bash -c 'snpe-dlc-quantize  \
    --input_dlc ../models/snpe/LibreYOLOXs_fp32.dlc \
    --input_list raw/filenames.txt \
    --output_dlc ../models/snpe/LibreYOLOXs_int8.dlc'

/qairt/2.41.0.251128/bin/x86_64-linux-clang/snpe-dlc-quantize: line 224: [: too many arguments
[INFO] InitializeStderr: DebugLog initialized.
[INFO] Processed command-line arguments
     0.6ms [  INFO ] Inferences will run in sync mode
Processing inference input(s):
raw/37f50879-f629-4031-aad7-11a6f275b00b.raw
raw/62caa7ce-10f9-4662-a28c-a11c73887828.raw
raw/08acaad7-ed4b-4332-b94a-b0ccda017b95.raw
raw/33d9a2d1-5f90-4434-98a3-3edb1b37cae4.raw
raw/891327c8-2523-4673-ba47-8b723c697992.raw
raw/12c65b90-6e09-4d96-be39-ab35ea592716.raw
raw/c9886ddb-5175-4ba4-a34d-0649697db970.raw
raw/e9f37b32-3873-4b90-8a6d-9005ca580e08.raw
raw/10f275be-4d70-467e-a0da-b19a11f60135.raw
raw/66c0b7d9-8bc1-4212-adf6-c72e92a1a5dd.raw
raw/ec319f5c-4cc9-450a-a5e3-fa395a42e9ef.raw
raw/7874b6aa-8815-46cd-905f-ba93937f0f9e.raw
raw/edebeb51-ea1f-4153-b5b5-490ae33c12b5.raw
raw/4a9c4fa2-7d1c-4d8e-a6dc-b07c2d1efd4d.raw
raw/5c307190-9262-43b0-b9d1-c4297b054fcb.raw
raw/50d4edb6-6fdb-49f5-96f5-6c8607a64503.raw
raw/f939457b-

### Inspecting the INT8 DLC

After quantization, `snpe-dlc-info` is used again to inspect the INT8 DLC. Compared to the FP32 output, you should observe that layer types now reflect quantized variants. The INT8 DLC is expected to be smaller than the FP32 DLC, but the actual reduction should be measured.

In [9]:
!bash -c 'snpe-dlc-info  \
    --input_dlc ../models/snpe/LibreYOLOXs_int8.dlc'

DLC info of: /workspace/models/snpe/LibreYOLOXs_int8.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: snpe-onnx-to-dlc; dump_inferred_model=False; dump_ir_optimizer_config_template=False; input_dim=None; converter_op_package_lib=; adjust_nms_features_dims=True; define_symbol=None; squash_box_decoder=True; inject_cast_for_gather=True; extract_color_transform=True; align_matmul_ranks=True; dumpIR=False; dump_custom_io_config_template=; dump_value_info=False; optimization_pass_mode_config=; preprocess_roi_pool_inputs=True; custom_op_config_paths=None; batch=None; use_quantize_v2=False; keep_int64_inputs=False; float_bias_bw=0; disable_batchnorm_folding=False; enable_strict_validation=False; disable_defer_loading=False; enable_framework_trace=False; dump_qairt_io_config_yaml=; preserve_io=[]; debug=-1; multi_time_steps_lstm=False; expand_lstm_op_structure=False; dump_pass_trace_info=

### Compiling the Graph for the Hexagon Tensor Processor (HTP)

Qualcomm® Snapdragon SoCs include a dedicated AI accelerator called the **Hexagon Tensor Processor (HTP)**. To fully leverage HTP hardware, SNPE can perform an **offline graph compilation** step that optimizes the DLC graph layout for a specific target SoC.

The `snpe-dlc-graph-prepare` command takes the INT8 DLC and the target SoC identifier — `sm7325`, corresponding to the **Snapdragon 778G** — and produces a pre-compiled DLC ready for on-device deployment with maximum HTP utilization.

In [10]:
!bash -c 'snpe-dlc-graph-prepare  \
    --input_dlc ../models/snpe/LibreYOLOXs_int8.dlc  \
    --output_dlc ../models/snpe/LibreYOLOXs_int8_sm7325.dlc \
    --htp_socs sm7325'

[INFO] InitializeStderr: DebugLog initialized.
[INFO] SNPE HTP Offline Prepare: Attempting to create cache for SM7325
[USER_INFO] Target device backend record identifier: HTP_V68_SM7325_2MB
[USER_INFO] No cache record in the DLC matches the target device (HTP_V68_SM7325_2MB). Creating a new record
[USER_INFO] Checking unsigned PD session
[INFO] Attempting to open dynamically linked lib: libHtpPrepare.so
[INFO] dlopen libHtpPrepare.so SUCCESS handle 0x5579f3083900
[INFO] Found Interface Provider (v2.31)
[USER_WARNING]  <W> Initializing HtpProvider
[USER_WARNING]  <W> HTP arch will be deprecated, please set SoC id instead.
[USER_WARNING]  <W> Performance Estimates unsupported on soc SM7325
[USER_WARNING]  <W> Cost Based unsupported on soc SM7325
[USER_INFO] Platform option not set
[USER_INFO] Created ctx=0x1 for Graph Id=0 backend=HTP SNPE Id=0x5579f2ebb7f8
[USER_INFO] Context [0x1] Setting priority to: default
[USER_INFO] Offline Prepare VTCM size(MB) selected = 0
[USER_INFO] Optimizati

### Inspecting the HTP-Optimized DLC

A final call to `snpe-dlc-info` confirms that the DLC has been compiled with HTP-specific optimizations for the **sm7325** SoC and is ready for on-device deployment.

In [11]:
!bash -c 'snpe-dlc-info \
    --input_dlc ../models/snpe/LibreYOLOXs_int8_sm7325.dlc'

DLC info of: /workspace/models/snpe/LibreYOLOXs_int8_sm7325.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: snpe-onnx-to-dlc; dump_inferred_model=False; dump_ir_optimizer_config_template=False; input_dim=None; converter_op_package_lib=; adjust_nms_features_dims=True; define_symbol=None; squash_box_decoder=True; inject_cast_for_gather=True; extract_color_transform=True; align_matmul_ranks=True; dumpIR=False; dump_custom_io_config_template=; dump_value_info=False; optimization_pass_mode_config=; preprocess_roi_pool_inputs=True; custom_op_config_paths=None; batch=None; use_quantize_v2=False; keep_int64_inputs=False; float_bias_bw=0; disable_batchnorm_folding=False; enable_strict_validation=False; disable_defer_loading=False; enable_framework_trace=False; dump_qairt_io_config_yaml=; preserve_io=[]; debug=-1; multi_time_steps_lstm=False; expand_lstm_op_structure=False; dump_pass_trac

By following these steps, the model is optimized for efficient inference on Qualcomm® platforms, leveraging hardware acceleration for real-time AI applications. This process ensures that the model is both accurate and performant, making it suitable for deployment in edge devices powered by Qualcomm® chipsets.